# Faruq-v3 — SGFR frozen residual synthesis
STB1 frozen + IGEM geometry residual, followed by AF2 frequency residual only after the geometry gate passes. Validation only; test is never extracted.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, shutil, subprocess, sys, time
from pathlib import Path

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/stb-igem-af2-frozen-synthesis'
if (REPO / '.git').is_dir():
    subprocess.run(['git','fetch','origin',BRANCH], cwd=REPO, check=True)
    subprocess.run(['git','checkout',BRANCH], cwd=REPO, check=True)
    subprocess.run(['git','reset','--hard',f'origin/{BRANCH}'], cwd=REPO, check=True)
else:
    if REPO.exists(): shutil.rmtree(REPO)
    command=['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)]
    for attempt in range(1,4):
        result=subprocess.run(command)
        if result.returncode == 0: break
        if REPO.exists(): shutil.rmtree(REPO)
        if attempt == 3: raise RuntimeError('Git clone gagal tiga kali')
        time.sleep(2)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)],check=True)
for key in list(sys.modules):
    if key == 'coffee_detector' or key.startswith('coffee_detector.'): sys.modules.pop(key,None)
sys.path.insert(0,str(REPO/'src'))
print('COMMIT:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip())

In [ ]:
import tarfile, torch
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root

assert torch.cuda.is_available(), 'Aktifkan T4 GPU.'
REQUIRED=(
 'bundles/faruq-development-v3-grouped.tar',
 'experiments/faruq-v3-breadth-screening-batch-v1/candidates/STB1/STB1_seed42/weights/best.pt',
 'experiments/faruq-v3-breadth-screening-batch-v1/candidates/STB1/val_reports/stb_seed42_screening.json',
)
PROJECT_ROOT=resolve_drive_project_root(required_relative_paths=REQUIRED)
ARCHIVE=require_project_artifact(PROJECT_ROOT,REQUIRED[0])
STB_CHECKPOINT=require_project_artifact(PROJECT_ROOT,REQUIRED[1])
STB_SUMMARY=require_project_artifact(PROJECT_ROOT,REQUIRED[2])
DATA_ROOT=Path('/content/faruq-development-v3-grouped')
if not (DATA_ROOT/'data.yaml').is_file():
    with tarfile.open(ARCHIVE,'r') as archive: archive.extractall('/content',filter='data')
GROUPED_SUMMARY=DATA_ROOT/'faruq_grouped_summary.json'
assert GROUPED_SUMMARY.is_file(), GROUPED_SUMMARY
assert not (DATA_ROOT/'test').exists(), 'Test tidak boleh tersedia.'
OUTPUT_ROOT=PROJECT_ROOT/'experiments/faruq-v3-sgfr-frozen-synthesis-v1'
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
STATIC_AUDIT=OUTPUT_ROOT/'static_audit.json'
print('GPU:',torch.cuda.get_device_name(0))
print('PROJECT:',PROJECT_ROOT)
print('OUTPUT:',OUTPUT_ROOT)

In [ ]:
from coffee_detector.sgfr.audit import static_sgfr_audit
result=static_sgfr_audit(REPO/'configs/coffee_fg/models/yolo26n-p3.yaml',STB_CHECKPOINT,STATIC_AUDIT)
print('DECISION:',result['decision'])
for arm,row in result['arms'].items():
    print(arm,row['decision'],'total=',row['parameters'],'trainable=',row['trainable_parameters'])
assert result['decision']=='PASS','STOP: static gate gagal; jangan training.'
print('PASS: raw box dan skor awal identik dengan STB1.')

## Stage 1 — optimizer control + geometry residual
Menjalankan SGC0 dan SGI1, masing-masing maksimal 20 epoch dan dapat resume dari Drive.

In [ ]:
command=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_sgfr_synthesis',
 '--data-root',str(DATA_ROOT),'--grouped-summary',str(GROUPED_SUMMARY),
 '--stb-summary',str(STB_SUMMARY),'--stb-checkpoint',str(STB_CHECKPOINT),
 '--static-audit',str(STATIC_AUDIT),'--output-root',str(OUTPUT_ROOT),
 '--stage','geometry','--seed','42','--device','0','--authorize-training']
print('MENJALANKAN:',' '.join(command),flush=True)
subprocess.run(command,cwd=REPO,check=True)

In [ ]:
import json, pandas as pd
from IPython.display import display
CORE_PATH=OUTPUT_ROOT/'val_reports/sgfr_geometry_seed42_decision.json'
core=json.loads(CORE_PATH.read_text())
rows=[{'model':'STB1',**core['reference']['STB1']},*[{'model':k,**v} for k,v in core['candidates'].items()]]
display(pd.DataFrame(rows).style.format({c:'{:.2%}' for c in ('macro_map50_95','bottom3_class_map50_95','worst_class_map50_95')}))
print('COMPARISONS:',json.dumps(core['comparisons'],indent=2))
print('DECISION:',core['decision'])
print('NEXT:',core['next_action'])
print('TEST OPENED:',core['test_opened'])
print('Kirim tabel dan keputusan. Jika FAIL, jangan menjalankan stage berikutnya.')

## Stage 2 — AF2 frequency residual
Jalankan hanya bila Stage 1 PASS. SGI1 direuse dan dibekukan; hanya 4.863 parameter frequency residual yang dilatih.

In [ ]:
core=json.loads((OUTPUT_ROOT/'val_reports/sgfr_geometry_seed42_decision.json').read_text())
if core['decision']!='PASS': raise RuntimeError('STOP: geometry stage FAIL.')
command=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_sgfr_synthesis',
 '--data-root',str(DATA_ROOT),'--grouped-summary',str(GROUPED_SUMMARY),
 '--stb-summary',str(STB_SUMMARY),'--stb-checkpoint',str(STB_CHECKPOINT),
 '--static-audit',str(STATIC_AUDIT),'--output-root',str(OUTPUT_ROOT),
 '--stage','frequency','--seed','42','--device','0','--authorize-training']
print('MENJALANKAN:',' '.join(command),flush=True)
subprocess.run(command,cwd=REPO,check=True)

In [ ]:
FINAL_PATH=OUTPUT_ROOT/'val_reports/sgfr_frequency_seed42_decision.json'
final=json.loads(FINAL_PATH.read_text())
rows=[{'model':'STB1',**final['reference']['STB1']},*[{'model':k,**v} for k,v in final['candidates'].items()]]
display(pd.DataFrame(rows).style.format({c:'{:.2%}' for c in ('macro_map50_95','bottom3_class_map50_95','worst_class_map50_95')}))
print('COMPARISONS:',json.dumps(final['comparisons'],indent=2))
print('DECISION:',final['decision'])
print('NEXT:',final['next_action'])
print('TEST OPENED:',final['test_opened'])
print('Kirim tabel dan keputusan. Jangan membuka test atau seed tambahan.')